<a href="https://colab.research.google.com/github/stefkong1982/netology.ru/blob/Master/DS_PROJECT/ds_proj_meth/CRISP_DM_titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Проект: Анализ выживаемости пассажиров Титаника  
**Автор:** [Ваше имя]  
**Дата:** [2023-11-15]  
**Цель:** Применить методы машинного обучения для прогнозирования выживаемости пассажиров Титаника на основе их характеристик, таких как пол, возраст, класс и другие факторы.

## 1. Понимание бизнеса (Business Understanding)  

**Цель проекта:**  
Прогнозирование выживаемости пассажиров Титаника на основе их характеристик для анализа факторов, влияющих на спасение людей в кораблекрушениях.  

**Постановка задачи:**  
Бинарная классификация пассажиров на выживших (1) и погибших (0) с использованием признаков: пол, возраст, класс каюты, количество родственников на борту и других.  

**Критерии успеха:**  
- **Метрики:**  
  - Accuracy (точность) – минимум 0.8  
  - Precision, Recall, F1-score (для дисбалансированных классов)  
  - AUC-ROC (оценка качества модели)  
- **Бизнес-результат:**  
  - Выявление ключевых факторов, повышающих шансы на выживание  
  - Возможность улучшения протоколов безопасности на судах

## 2. Понимание данных (Data Understanding)

### Импорт библиотек

In [1]:
# Импорт библиотек для работы с данными
import polars as pl  # Основная библиотека для обработки и анализа данных

# Импорт библиотек для визуализации (понадобятся позже)
import matplotlib.pyplot as plt  # Построение графиков
import seaborn as sns  # Улучшенная визуализация данных

# Импорт библиотек для машинного обучения (понадобятся позже)
from sklearn.model_selection import train_test_split  # Разделение данных на train/test
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score  # Метрики оценки

### Загрузка и первичный анализ данных


In [6]:
# Настройка отображения
pl.Config.set_tbl_cols(-1)  # Показать все столбцы
pl.Config.set_tbl_width_chars(180)  # Ширина таблицы
pl.Config.set_tbl_rows(5)  # Ограничить количество выводимых строк

polars.config.Config

In [7]:
# Загрузка данных Titanic из seaborn
import seaborn as sns
df = sns.load_dataset('titanic')

In [8]:
# Конвертация pandas DataFrame в polars DataFrame
df_titanic = pl.from_pandas(df)

In [9]:
# Вывод данных для первичного анализа
print("Данные обучающей выборки:")
print(df_titanic)

Данные обучающей выборки:
shape: (891, 15)
┌──────────┬────────┬────────┬──────┬───────┬───────┬─────────┬──────────┬───────┬───────┬────────────┬──────┬─────────────┬───────┬───────┐
│ survived ┆ pclass ┆ sex    ┆ age  ┆ sibsp ┆ parch ┆ fare    ┆ embarked ┆ class ┆ who   ┆ adult_male ┆ deck ┆ embark_town ┆ alive ┆ alone │
│ ---      ┆ ---    ┆ ---    ┆ ---  ┆ ---   ┆ ---   ┆ ---     ┆ ---      ┆ ---   ┆ ---   ┆ ---        ┆ ---  ┆ ---         ┆ ---   ┆ ---   │
│ i64      ┆ i64    ┆ str    ┆ f64  ┆ i64   ┆ i64   ┆ f64     ┆ str      ┆ cat   ┆ str   ┆ bool       ┆ cat  ┆ str         ┆ str   ┆ bool  │
╞══════════╪════════╪════════╪══════╪═══════╪═══════╪═════════╪══════════╪═══════╪═══════╪════════════╪══════╪═════════════╪═══════╪═══════╡
│ 0        ┆ 3      ┆ male   ┆ 22.0 ┆ 1     ┆ 0     ┆ 7.25    ┆ S        ┆ Third ┆ man   ┆ true       ┆ null ┆ Southampton ┆ no    ┆ false │
│ 1        ┆ 1      ┆ female ┆ 38.0 ┆ 1     ┆ 0     ┆ 71.2833 ┆ C        ┆ First ┆ woman ┆ false      ┆ C    ┆ 

```
Данные обучающей выборки:
shape: (891, 15)
┌──────────┬────────┬────────┬──────┬───────┬───────┬─────────┬──────────┬───────┬───────┬────────────┬──────┬─────────────┬───────┬───────┐
│ survived ┆ pclass ┆ sex    ┆ age  ┆ sibsp ┆ parch ┆ fare    ┆ embarked ┆ class ┆ who   ┆ adult_male ┆ deck ┆ embark_town ┆ alive ┆ alone │
│ ---      ┆ ---    ┆ ---    ┆ ---  ┆ ---   ┆ ---   ┆ ---     ┆ ---      ┆ ---   ┆ ---   ┆ ---        ┆ ---  ┆ ---         ┆ ---   ┆ ---   │
│ i64      ┆ i64    ┆ str    ┆ f64  ┆ i64   ┆ i64   ┆ f64     ┆ str      ┆ cat   ┆ str   ┆ bool       ┆ cat  ┆ str         ┆ str   ┆ bool  │
╞══════════╪════════╪════════╪══════╪═══════╪═══════╪═════════╪══════════╪═══════╪═══════╪════════════╪══════╪═════════════╪═══════╪═══════╡
│ 0        ┆ 3      ┆ male   ┆ 22.0 ┆ 1     ┆ 0     ┆ 7.25    ┆ S        ┆ Third ┆ man   ┆ true       ┆ null ┆ Southampton ┆ no    ┆ false │
│ 1        ┆ 1      ┆ female ┆ 38.0 ┆ 1     ┆ 0     ┆ 71.2833 ┆ C        ┆ First ┆ woman ┆ false      ┆ C    ┆ Cherbourg   ┆ yes   ┆ false │
│ 1        ┆ 3      ┆ female ┆ 26.0 ┆ 0     ┆ 0     ┆ 7.925   ┆ S        ┆ Third ┆ woman ┆ false      ┆ null ┆ Southampton ┆ yes   ┆ true  │
│ …        ┆ …      ┆ …      ┆ …    ┆ …     ┆ …     ┆ …       ┆ …        ┆ …     ┆ …     ┆ …          ┆ …    ┆ …           ┆ …     ┆ …     │
│ 1        ┆ 1      ┆ male   ┆ 26.0 ┆ 0     ┆ 0     ┆ 30.0    ┆ C        ┆ First ┆ man   ┆ true       ┆ C    ┆ Cherbourg   ┆ yes   ┆ true  │
│ 0        ┆ 3      ┆ male   ┆ 32.0 ┆ 0     ┆ 0     ┆ 7.75    ┆ Q        ┆ Third ┆ man   ┆ true       ┆ null ┆ Queenstown  ┆ no    ┆ true  │
└──────────┴────────┴────────┴──────┴───────┴───────┴─────────┴──────────┴───────┴───────┴────────────┴──────┴─────────────┴───────┴───────┘
```

Описание бинарных переменных

1. **survived**: Целевая переменная (1 - выжил, 0 - погиб)
2. **sex**: Пол пассажира (male/female)
3. **adult_male**: Взрослый мужчина (True/False)
4. **alive**: Факт выживания (yes/no)
5. **alone**: Путешествовал один (True/False)